In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time
from functools import partial
from scipy.optimize import curve_fit

# ---------------- CONFIGURATION ----------------
BETA = 0.7          # coupling
N_MAX = 2           # flux cutoff
ITERATIONS = 10     # 2^ITERATIONS x 2^ITERATIONS lattice
THETA_POINTS = 20   # θ resolution for scan

print(f"--- CONFIGURATION ---")
print(f"System       : 2D U(1) Lattice Gauge Theory")
print(f"Beta         : {BETA}")
print(f"Flux Cutoff  : {N_MAX}")
print(f"Lattice Size : {2**ITERATIONS} x {2**ITERATIONS}")
try:
    print(f"Backend      : {jax.devices()[0].device_kind}")
except:
    print("Backend      : CPU (JAX)")
print("-" * 40)


# ------------- 1. COMPLEX TENSOR CONSTRUCTION -------------

def create_complex_gauge_tensor(beta, theta, N_max):
    """
    Rank-4 tensor for 2D U(1) with a θ-term.
    T[i,j,k,l] = Villain weight * exp(i θ n_plaq)
    Uses complex128 / float64 for high precision.
    """
    D = 2 * N_max + 1
    T = jnp.zeros((D, D, D, D), dtype=jnp.complex128)
    fluxes = jnp.arange(-N_max, N_max + 1, dtype=jnp.float64)

    for i_idx, n_i in enumerate(fluxes):
        for j_idx, n_j in enumerate(fluxes):
            for k_idx, n_k in enumerate(fluxes):
                for l_idx, n_l in enumerate(fluxes):
                    # Bianchi constraint (flux conservation at vertex)
                    if n_i + n_j - n_k - n_l == 0:
                        n_plaq = (n_i + n_j + n_k + n_l) / 4.0

                        # Villain-like Gaussian suppression
                        real_weight = jnp.exp(-beta * (2.0 * jnp.pi * n_plaq)**2 / 2.0)

                        # θ-term phase
                        phase = jnp.exp(1j * theta * n_plaq)

                        T = T.at[i_idx, j_idx, k_idx, l_idx].set(real_weight * phase)
    return T


# ------------- 2. TRG CORE (JIT COMPILED) -------------

@partial(jax.jit, static_argnums=(1,))
def trg_step_complex(T, chi_max):
    """
    One TRG coarse-graining step for a complex tensor T.
    chi_max is the TRG bond dimension.
    """
    D = T.shape[0]

    # Horizontal decomposition
    M_h = T.transpose(0, 2, 3, 1).reshape(D*D, D*D)
    U_h, S_h, Vdag_h = jnp.linalg.svd(M_h, full_matrices=False)
    chi_h = jnp.minimum(S_h.shape[0], chi_max)

    sqrt_S_h = jnp.sqrt(S_h[:chi_h])
    T1 = (U_h[:, :chi_h] @ jnp.diag(sqrt_S_h)).reshape(D, D, chi_h)
    T2 = (jnp.diag(sqrt_S_h) @ Vdag_h[:chi_h, :]).reshape(chi_h, D, D)

    # Vertical decomposition
    M_v = T.reshape(D*D, D*D)
    U_v, S_v, Vdag_v = jnp.linalg.svd(M_v, full_matrices=False)
    chi_v = jnp.minimum(S_v.shape[0], chi_max)

    sqrt_S_v = jnp.sqrt(S_v[:chi_v])
    T3 = (U_v[:, :chi_v] @ jnp.diag(sqrt_S_v)).reshape(D, D, chi_v)
    T4 = (jnp.diag(sqrt_S_v) @ Vdag_v[:chi_v, :]).reshape(chi_v, D, D)

    # Contraction
    T_new = jnp.einsum('ika,blj,ijc,dkl->acbd', T1, T2, T3, T4)

    # Normalize and track norm
    norm = jnp.linalg.norm(T_new)
    T_new = T_new / norm

    return T_new, jnp.log(norm)


def compute_free_energy(beta, theta, chi_max):
    """
    Full TRG flow → free energy density f(θ) = - (1/V) log Z.
    """
    T = create_complex_gauge_tensor(beta, theta, N_MAX)
    log_Z_accum = 0.0

    for _ in range(ITERATIONS):
        T, log_norm = trg_step_complex(T, chi_max)
        log_Z_accum += log_norm

    Z_final = jnp.einsum('ijkl->', T)
    log_Z_final = jnp.log(Z_final) + log_Z_accum

    Volume = 4**ITERATIONS
    f = -jnp.real(log_Z_final) / Volume
    return f


# ------------- 3. χ_max CONVERGENCE SCAN -------------

def chi_scan(beta, theta, chi_list):
    print(f"\n=== χ_max convergence scan at β={beta}, θ={theta} ===")
    f_vals = []
    times = []

    # Warmup JIT
    _ = compute_free_energy(beta, theta, chi_list[0]).block_until_ready()

    for chi in chi_list:
        start = time.time()
        f = compute_free_energy(beta, theta, chi)
        f.block_until_ready()
        dt = time.time() - start
        f_vals.append(float(f))
        times.append(dt)
        print(f"χ_max={chi:3d} | f(θ)={f:.10e} | time={dt:6.3f}s")

    plt.figure(figsize=(6,4))
    plt.plot(chi_list, f_vals, 'o-')
    plt.xlabel("χ_max")
    plt.ylabel("f(θ)")
    plt.title(f"Convergence vs χ_max (β={beta}, θ={theta})")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return chi_list, f_vals, times


# ------------- 4. θ-SCAN -------------

def theta_scan(beta, chi_max, n_theta=THETA_POINTS):
    print(f"\n=== θ-scan at β={beta}, χ_max={chi_max} ===")
    thetas = np.linspace(0.0, 2.0*np.pi, n_theta)
    energies = []
    times = []

    # Warmup
    _ = compute_free_energy(beta, 0.0, chi_max).block_until_ready()

    for theta in thetas:
        start = time.time()
        f = compute_free_energy(beta, theta, chi_max)
        f.block_until_ready()
        dt = time.time() - start
        energies.append(float(f))
        times.append(dt)
        print(f"θ={theta:5.2f} | f(θ)={f:.10e} | {dt*1000:6.1f} ms")

    # Plot and cosine fit
    plt.figure(figsize=(8,5))
    plt.plot(thetas, energies, 'o', color='crimson', label=f'TRG (χ={chi_max})')

    def cosine_model(x, a, b):
        return a * np.cos(x) + b

    try:
        popt, _ = curve_fit(cosine_model, thetas, energies)
        fit_curve = cosine_model(thetas, *popt)
        plt.plot(thetas, fit_curve, 'k--', label='cosine fit')
        print(f"\nCosine fit parameters: a={popt[0]:.4e}, b={popt[1]:.4e}")
        resid = np.array(energies) - fit_curve
        print(f"Max |residual| ≈ {np.max(np.abs(resid)):.3e}")
    except Exception as e:
        print("Cosine fit failed:", e)

    plt.xlabel("θ")
    plt.ylabel("f(θ)")
    plt.title(f"2D U(1) TRG: f(θ) at β={beta}, χ_max={chi_max}")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"\nTotal scan time: {sum(times):.2f}s")
    return thetas, energies, times


# ------------- 5. RUN TESTS -------------

# (1) χ_max convergence at a single θ
chi_list = [8, 12, 16, 24, 32]
chi_scan(BETA, theta=np.pi/2, chi_list=chi_list)

# (2) θ-scan at a χ_max you trust (e.g. 32)
theta_scan(BETA, chi_max=32, n_theta=THETA_POINTS)


--- CONFIGURATION ---
System       : 2D U(1) Lattice Gauge Theory
Beta         : 0.7
Flux Cutoff  : 2
Lattice Size : 1024 x 1024
Backend      : cpu
----------------------------------------

=== χ_max convergence scan at β=0.7, θ=1.5707963267948966 ===


/tmp/ipython-input-2556516546.py:36: UserWarning: Explicitly requested dtype <class 'jax.numpy.complex128'> requested in zeros is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  T = jnp.zeros((D, D, D, D), dtype=jnp.complex128)
/usr/local/lib/python3.12/dist-packages/jax/_src/numpy/lax_numpy.py:5943: UserWarning: Explicitly requested dtype <class 'jax.numpy.float64'> requested in arange is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return _arange(start, stop=stop, step=step, dtype=dtype,


IndexError: Array slice indices must have static start/stop/step to be used with NumPy indexing syntax. Found slice(None, JitTracer<~int32[]>, None). To index a statically sized array at a dynamic position, try lax.dynamic_slice/dynamic_update_slice (JAX does not support dynamically sized arrays within JIT compiled functions).